# 목표

뉴스의 카테고리 예측

In [1]:
# %load_ext colablinter

# 데이터 파악

In [2]:
import torch
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import gc
import os


# 시각화 관련 설정
try:
    plt.rcParams['font.family'] = 'Apple SD Gothic Neo'
except:
    try:
        plt.rcParams['font.family'] = 'NanumGothic'
    except:
        plt.rcParams['font.family'] = 'AppleGothic'

plt.rcParams['axes.unicode_minus'] = False
fm._load_fontmanager(try_read_cache=False)


# 디바이스 설정
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    DEVICE = torch.device("mps") # 맥 GPU
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda:0") # 윈도우 GPU
else:
    DEVICE = torch.device("cpu") # CPU


# 캐시 지우기 함수 생성
def clean_cache():
    gc.collect()
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        torch.mps.empty_cache()
    elif torch.cuda.is_available():
        torch.cuda.empty_cache()

# MallocStackLogging 에러 출력 방지
os.environ.pop("MallocStackLogging", None)
os.environ.pop("MallocStackLoggingNoCompact", None)
os.environ.pop("DYLD_INSERT_LIBRARIES", None)


# # 로그
# import logging

# def init_logger() -> logging.Logger:
#     logging.basicConfig(
#         format="%(asctime)s [%(levelname)s] (%(filename)s:%(lineno)d) - %(message)s",
#         datefmt="%Y-%m-%d %H:%M:%S",
#         level=logging.INFO,
#         encoding="utf-8",
#     )
#     return logging.getLogger("")

# logger = init_logger()

Matplotlib is building the font cache; this may take a moment.


In [4]:
ROOT_DIR = os.getcwd()
DATA_DIR = os.path.join(ROOT_DIR, "data")
TRAIN_DIR = os.path.join(DATA_DIR, "20news-bydate-train")
TEST_DIR = os.path.join(DATA_DIR, "20news-bydate-test")

In [4]:
# 파일 개수 세기
from glob import glob

train_category_list = os.listdir(TRAIN_DIR)
test_category_list = os.listdir(TEST_DIR)

train_path_list = [path_ for path_ in glob(os.path.join(TRAIN_DIR, "**", "**")) 
               if os.path.isfile(path_)]
test_path_list = [path_ for path_ in glob(os.path.join(TEST_DIR, "**", "**")) 
              if os.path.isfile(path_)]

print(f"· 학습 데이터: {len(train_path_list)}개")
print(f"· 테스트 데이터: {len(test_path_list)}개")

· 학습 데이터: 11314개
· 테스트 데이터: 7532개


In [5]:
train_num_list = [path_.split("/")[-1] for path_ in train_path_list]
test_num_list = [path_.split("/")[-1] for path_ in test_path_list]

intersection = set(train_num_list) & set(test_num_list)

print(f"· 학습 - 테스트 데이터 겹치는 번호: {len(intersection)}개")

· 학습 - 테스트 데이터 겹치는 번호: 1294개


In [6]:
import chardet
from collections import Counter

def detect_encoding(file_path):
    """파일의 인코딩을 감지"""
    with open(file_path, 'rb') as f:
        raw_data = f.read()
    result = chardet.detect(raw_data)
    return result['encoding'], result['confidence']

# 모든 파일의 인코딩 정보 수집
encoding_info = {}
encoding_counter = Counter()

all_paths = train_path_list + test_path_list

for i, path in enumerate(all_paths):   
    try:
        encoding, confidence = detect_encoding(path)
        encoding_info[path] = {'encoding': encoding, 'confidence': confidence}
        encoding_counter[encoding] += 1
    except Exception as e:
        print(f"오류 발생 ({path}): {e}")

print("[인코딩 분포]")
for encoding, count in encoding_counter.most_common():
    percentage = (count / len(all_paths)) * 100
    print(f"· {encoding}: {count}개 ({percentage:.2f}%)")

[인코딩 분포]
· ascii: 18767개 (99.58%)
· ISO-8859-1: 58개 (0.31%)
· Windows-1252: 7개 (0.04%)
· MacRoman: 6개 (0.03%)
· None: 3개 (0.02%)
· HZ-GB-2312: 2개 (0.01%)
· Johab: 1개 (0.01%)
· ISO-2022-JP: 1개 (0.01%)
· ISO-8859-7: 1개 (0.01%)


In [7]:
def get_text(path):
    """감지된 인코딩으로 파일 읽기"""
    # encoding_info에 정보가 있으면 사용
    encoding = encoding_info.get(path, {}).get('encoding', 'latin-1')
    
    # None이면 latin-1 사용 (모든 바이트값을 읽을 수 있음)
    if encoding is None:
        encoding = 'latin-1'
    
    try:
        with open(path, "r", encoding=encoding) as f:
            return f.read()
    except:
        # 실패시 latin-1로 폴백 (절대 실패하지 않음)
        with open(path, "r", encoding="latin-1") as f:
            return f.read()

In [8]:
print("[겹치는 번호 오류 점검]")

train_dup_dict = {path_.split("/")[-1]: path_.split("/")[-2] for path_ in train_path_list 
               if path_.split("/")[-1] in intersection}

test_dup_dict = {path_.split("/")[-1]: path_.split("/")[-2] for path_ in test_path_list
               if path_.split("/")[-1] in intersection}


dup_path_list = []

for file_num in intersection:
    train_path = os.path.join(TRAIN_DIR, train_dup_dict[file_num], file_num)
    test_path = os.path.join(TEST_DIR, test_dup_dict[file_num], file_num)

    train_intersection = get_text(train_path).strip()
    test_intersection = get_text(test_path).strip()

    if train_intersection == test_intersection:
        dup_path_list.append(train_path)

print(f"· 내용 같은 파일: {len(dup_path_list)}개")

[겹치는 번호 오류 점검]
· 내용 같은 파일: 0개


단순히 번호만 겹쳤던 것으로 확인된다.

In [9]:
train_category_set = set(train_category_list)
test_category_set = set(test_category_list)

only_in_train = train_category_set - test_category_set
only_in_test = test_category_set - train_category_set

print("[카테고리 결손 파악]")
print(f"· 학습 데이터에만 있는 카테고리: {len(only_in_train)}개")
print(f"· 테스트 데이터에만 있는 카테고리: {len(only_in_test)}개")

[카테고리 결손 파악]
· 학습 데이터에만 있는 카테고리: 0개
· 테스트 데이터에만 있는 카테고리: 0개


In [10]:
CATEGORY_LIST = os.listdir(TRAIN_DIR)

In [11]:
print("[카테고리별 데이터 개수]")
i = 0

for category in CATEGORY_LIST:

    i += 1
    path_list = [path_ for path_ in glob(os.path.join(TRAIN_DIR, category, "*")) 
                 if os.path.isfile(path_)]
    
    print(f"{i}) {category}: {len(path_list)}개")

[카테고리별 데이터 개수]
1) talk.politics.mideast: 564개
2) rec.autos: 594개
3) comp.sys.mac.hardware: 578개
4) alt.atheism: 480개
5) rec.sport.baseball: 597개
6) comp.os.ms-windows.misc: 591개
7) rec.sport.hockey: 600개
8) sci.crypt: 595개
9) sci.med: 594개
10) talk.politics.misc: 465개
11) rec.motorcycles: 598개
12) comp.windows.x: 593개
13) comp.graphics: 584개
14) comp.sys.ibm.pc.hardware: 590개
15) sci.electronics: 591개
16) talk.politics.guns: 546개
17) sci.space: 593개
18) soc.religion.christian: 599개
19) misc.forsale: 585개
20) talk.religion.misc: 377개


- talk.religion.misc 카테고리가 좀 적네

- 상위 하위 카테고리로 구성되어 있으니, 카테고리도 더 나눠보자.

    - misc는 잡동사니라는 뜻으로, 주요 카테고리가 아니라는 뜻이다.

    - 끝의 카테고리를 main이라고 하나 만들어서 misc와 분리하자.

In [12]:
TRAIN_DICT = dict()

for category in CATEGORY_LIST:
    for path_ in glob(os.path.join(TRAIN_DIR, category, "*")):

        if os.path.isfile(path_):
            
            TRAIN_DICT[path_] = dict()
            TRAIN_DICT[path_]["category"] = category

            splited_category = category.split(".")
            if splited_category[-1] != "misc":
                splited_category.append("main")

            for i in range(0, len(splited_category)):
                TRAIN_DICT[path_][i] = splited_category[i]

In [13]:
for _, dictionary in TRAIN_DICT.items():
    for i in range(1, len(dictionary) - 1):
        if dictionary[i] == "main":
            dictionary[5] = "main"
            del dictionary[i]
            continue

        elif dictionary[i] == "misc":
            dictionary[5] = "misc"
            del dictionary[i]

In [14]:
print("[TRAIN_DICT 예시 출력]")
for key, value in TRAIN_DICT.items():
    print(key, "\n: ", value)
    break

[TRAIN_DICT 예시 출력]
/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.politics.mideast/75895 
:  {'category': 'talk.politics.mideast', 0: 'talk', 1: 'politics', 2: 'mideast', 5: 'main'}


In [15]:
# import json

# train_json_path = os.path.join(DATA_DIR, "train.json")

# with open(train_json_path, "w", encoding="utf-8") as f:
#     json.dump(TRAIN_DICT, f, indent=4)


# test_json_path = os.path.join(DATA_DIR, "test.json")

# with open(test_json_path, "w", encoding="utf-8") as f:
#     json.dump(TEST_DICT, f, indent=4)

In [16]:
count_by_category_dict = {
              0: {},
              1: {},
              2: {},
              3: {},
              4: {},
              5: {}
              }


for category_dict in TRAIN_DICT.values():
    for key, value in category_dict.items():
        if key != "category":
            if value not in count_by_category_dict[key].keys():
                count_by_category_dict[key][value] = 1
            else:
                count_by_category_dict[key][value] += 1

In [17]:
import plotly.graph_objects as go

# 데이터
data = count_by_category_dict

nodes_by_level_and_name = {}
nodes = {}
edges = set()
node_id = 0

root_key = ('ROOT', 0)
nodes[root_key] = {'id': node_id, 'label': 'ROOT', 'level': 0, 'count': len(TRAIN_DICT)}
nodes_by_level_and_name[(0, 'ROOT')] = root_key
node_id += 1

for level, name_count_dict in data.items():
    current_level = level + 1
    for name, count in name_count_dict.items():
        node_key = (name, current_level)
        if node_key not in nodes:
            nodes[node_key] = {
                'id': node_id,
                'label': name,
                'level': current_level,
                'count': count,
            }
            nodes_by_level_and_name[(current_level, name)] = node_key
            node_id += 1

for _, category_dict in TRAIN_DICT.items():
    ordered_path = sorted(
        (int(k), v) for k, v in category_dict.items() if k != 'category'
    )
    parent_key = root_key
    for level_idx, name in ordered_path:
        current_level = level_idx + 1
        node_key = nodes_by_level_and_name.get((current_level, name))
        if node_key is None:
            continue
        edges.add((parent_key, node_key))
        parent_key = node_key

levels = {}
for node_key, node in nodes.items():
    levels.setdefault(node['level'], []).append(node_key)


# 노드의 최대 자식 깊이를 계산하는 함수
def get_max_depth(node_key, edges, memo=None):
    """
    해당 노드에서 시작하여 도달할 수 있는 최대 깊이를 반환합니다.
    """
    if memo is None:
        memo = {}
    
    if node_key in memo:
        return memo[node_key]
    
    # 자식 노드 찾기
    children = [child for parent, child in edges if parent == node_key]
    
    if not children:
        memo[node_key] = 0
        return 0
    
    # 자식들의 최대 깊이 + 1
    max_child_depth = max(get_max_depth(child, edges, memo) for child in children)
    memo[node_key] = max_child_depth + 1
    
    return memo[node_key]


# 부모-자식 관계를 기반으로 노드 정렬 함수
def sort_nodes_by_hierarchy(node_keys, edges, nodes):
    """
    부모-자식 관계를 고려하여 노드를 정렬합니다.
    깊이가 깊은 노드를 왼쪽에 배치하고 (레벨 5 제외),
    같은 부모를 가진 노드들끼리 그룹화합니다.
    """
    if not node_keys:
        return []
    
    # 각 노드의 부모들 찾기
    node_parents = {}
    for parent, child in edges:
        if child in node_keys:
            if child not in node_parents:
                node_parents[child] = []
            node_parents[child].append(parent)
    
    # 부모가 없는 노드들 (루트 레벨)
    if not node_parents:
        return sorted(node_keys, key=lambda x: nodes[x]['label'])
    
    # 각 노드의 최대 깊이 계산
    depth_memo = {}
    for node_key in node_keys:
        get_max_depth(node_key, edges, depth_memo)
    
    # 부모별로 자식 노드 그룹화
    parent_to_children = {}
    for node in node_keys:
        parents = tuple(sorted(node_parents.get(node, [])))
        if parents not in parent_to_children:
            parent_to_children[parents] = []
        parent_to_children[parents].append(node)
    
    # 각 그룹 내에서 정렬
    # 레벨 5가 아니면: 깊이가 깊은 것 -> 알파벳순
    # 레벨 5이면: 알파벳순만
    for parents in parent_to_children:
        children = parent_to_children[parents]
        current_level = nodes[children[0]]['level']
        
        if current_level == 6:  # 레벨 5 (0-indexed이므로 6)
            # 레벨 5는 알파벳순으로만 정렬
            parent_to_children[parents].sort(key=lambda x: nodes[x]['label'])
        else:
            # 다른 레벨은 깊이 우선 -> 알파벳순
            parent_to_children[parents].sort(
                key=lambda x: (-depth_memo.get(x, 0), nodes[x]['label'])
            )
    
    # 부모의 x 좌표 순서대로 자식들을 배치
    sorted_nodes = []
    
    # 첫 번째 레벨은 부모의 x 좌표가 없으므로 깊이 -> 알파벳순으로
    if all(p not in nodes or 'x' not in nodes[p] for parents in parent_to_children.keys() for p in parents):
        sorted_parent_groups = sorted(
            parent_to_children.keys(),
            key=lambda parents: (
                -max(depth_memo.get(child, 0) for child in parent_to_children[parents]),
                min(nodes[child]['label'] for child in parent_to_children[parents])
            )
        )
    else:
        # 부모의 x 좌표를 기준으로 정렬
        def get_parent_x(parents):
            if not parents:
                return 0
            parent_xs = [nodes[p].get('x', 0) for p in parents if p in nodes]
            return sum(parent_xs) / len(parent_xs) if parent_xs else 0
        
        sorted_parent_groups = sorted(parent_to_children.keys(), key=get_parent_x)
    
    for parents in sorted_parent_groups:
        sorted_nodes.extend(parent_to_children[parents])
    
    return sorted_nodes


# 레벨별 노드 배치 (계층 구조 기반 정렬 적용)
for level, node_keys in levels.items():
    # 계층 구조 기반 정렬 적용
    sorted_node_keys = sort_nodes_by_hierarchy(node_keys, edges, nodes)
    
    num_nodes = len(sorted_node_keys)
    y = -level * 2
    for i, key in enumerate(sorted_node_keys):
        if num_nodes == 1:
            x = 0
        else:
            spacing = min(15, 50 / num_nodes)
            x = (i - (num_nodes - 1) / 2) * spacing
        nodes[key]['x'] = x
        nodes[key]['y'] = y

edge_x = []
edge_y = []
for parent, child in edges:
    edge_x.extend([nodes[parent]['x'], nodes[child]['x'], None])
    edge_y.extend([nodes[parent]['y'], nodes[child]['y'], None])

edge_trace = go.Scatter(
    x=edge_x,
    y=edge_y,
    mode='lines',
    line=dict(width=1.5, color='#cccccc'),
    hoverinfo='none'
)

node_colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c', '#e67e22']
node_x = []
node_y = []
node_labels = []
node_counts = []
node_color = []
node_sizes = []

for n in nodes.values():
    node_x.append(n['x'])
    node_y.append(n['y'])
    node_labels.append(n['label'])
    node_counts.append(str(n.get('count', 0)))
    node_color.append(node_colors[n['level'] % len(node_colors)])
    base_size = 14 if n['level'] == 0 else 10
    scale = 0 if n.get('count') is None else max(0, n['count']) ** 0.5
    node_sizes.append(base_size + scale)

parent_counts = {}
for parent, child in edges:
    parent_counts[child] = parent_counts.get(child, 0) + 1

hover_text = []
for node_key, n in nodes.items():
    parent_count = parent_counts.get(node_key, 0)
    base_text = f"{n['label']}<br>레벨: {n['level']}<br>개수: {n.get('count', 0)}"
    if parent_count > 1:
        hover_text.append(base_text + f"<br>부모 노드: {parent_count}개")
    else:
        hover_text.append(base_text)

node_trace = go.Scatter(
    x=node_x,
    y=node_y,
    mode='markers',
    marker=dict(
        size=node_sizes,
        color=node_color,
        line=dict(width=2, color='white')
    ),
    hoverinfo='text',
    hovertext=hover_text
)

count_text_trace = go.Scatter(
    x=node_x,
    y=node_y,
    mode='text',
    text=node_counts,
    textposition='middle center',
    textfont=dict(size=10, color='white', family='Arial'),
    hoverinfo='none'
)

label_text_trace = go.Scatter(
    x=node_x,
    y=[y + 0.5 for y in node_y],
    mode='text',
    text=node_labels,
    textposition='top center',
    textfont=dict(size=10, color='black', family='Arial'),
    hoverinfo='none'
)

layout = go.Layout(
    title=dict(text='카테고리 계층 구조 (중복 노드 병합)', font=dict(size=20)),
    showlegend=False,
    hovermode='closest',
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    plot_bgcolor='#ffffff',
    paper_bgcolor='#ffffff',
    margin=dict(l=40, r=40, t=60, b=40),
    height=900,
)

fig = go.Figure(
    data=[edge_trace, node_trace, count_text_trace, label_text_trace],
    layout=layout,
)
fig.show()


In [18]:
sub_category_list = list()
for category in CATEGORY_LIST:
    sub_category_list.extend(category.split(".")[1:])


multi_list = list()
for sub_category in set(sub_category_list):
    if sub_category_list.count(sub_category) > 1:
        multi_list.append(sub_category)


multi_dict = dict()
for category in CATEGORY_LIST:
    for multi_class in multi_list:
        if multi_class in category.split(".")[1:]:
            if multi_class in multi_dict.keys():
                multi_dict[multi_class].append(category)
            else:
                multi_dict[multi_class] = [category]


print("[여러 요소가 있는 카테고리]")
for key, value in multi_dict.items():
    print(f"· {key}: {value}")

[여러 요소가 있는 카테고리]
· politics: ['talk.politics.mideast', 'talk.politics.misc', 'talk.politics.guns']
· sys: ['comp.sys.mac.hardware', 'comp.sys.ibm.pc.hardware']
· hardware: ['comp.sys.mac.hardware', 'comp.sys.ibm.pc.hardware']
· sport: ['rec.sport.baseball', 'rec.sport.hockey']
· misc: ['comp.os.ms-windows.misc', 'talk.politics.misc', 'talk.religion.misc']
· religion: ['soc.religion.christian', 'talk.religion.misc']


1. 각 카테고리를 합쳤을 때는 불균형이 그렇게 심하게 보이지 않지만, 쪼개서 노드로 보니 결과가 다르다.

2. religion이라는 카테고리는 2개의 메인 카테고리에 딸려 있어 복잡하다. 계속 주시해야 하는 파트라는 생각이 든다.

3. 가장 하위 카테고리가 misc인 카테고리와 아닌 카테고리가 있는데 그것들만을 기준으로 보면 불균형은 아주 심하다.

4. hardware는 다른 깊이에 존재한다.

# 실험 계획

1. 주 카테고리 먼저 -> 한 차례 씩 밑으로 내려가며 학습. 각 단계마다 카테고리 불균형 해소 진행.

2. 주 카테고리와 가장 하위 카테고리(main, misc 제외) 학습

3. 그냥 한 번에 학습

이 세 개로 나누어서 비교한다.

# 데이터 전처리 

## 1. 테스트 데이터 형식 조정

In [19]:
TEST_DICT = dict()

for category in CATEGORY_LIST:
    for path_ in glob(os.path.join(TEST_DIR, category, "*")):

        if os.path.isfile(path_):
            
            TEST_DICT[path_] = dict()
            TEST_DICT[path_]["category"] = category

            splited_category = category.split(".")
            if splited_category[-1] != "misc":
                splited_category.append("main")

            for i in range(0, len(splited_category)):
                TEST_DICT[path_][i] = splited_category[i]


for _, dictionary in TEST_DICT.items():
    for i in range(1, len(dictionary) - 1):
        if dictionary[i] == "main":
            dictionary[5] = "main"
            del dictionary[i]
            continue

        elif dictionary[i] == "misc":
            dictionary[5] = "misc"
            del dictionary[i]

## 2. 텍스트 정제

In [20]:
import random

rand_idx = random.randint(1, len(TRAIN_DICT.keys()))
rand_key = list(TRAIN_DICT.keys())[rand_idx]

sample_key = rand_key.split("/")[-2]
sample_text = get_text(rand_key)

print(f"[샘플 출력]")
print(f"· 카테고리: {sample_key}")
print("· 본문", "="*40)
print(sample_text)

[샘플 출력]
· 카테고리: comp.os.ms-windows.misc
· 본문 ========================================
Subject: Re: Windows for WorkGroups and LAN Workplace
From: Flint.Waters@uwyo.edu (Flint Waters)
Distribution: comp.os.ms-windows.apps,comp.os.ms-windows.misc,comp.os.ms-,world
Organization: University of Wyoming
Nntp-Posting-Host: sheriff.uwyo.edu
Lines: 13


>Now does anyone know if it is possible to use W4WG and Lan Workplace
>for DOS at the same time. 

>ie Can I access a file on another PC while being logged on to the
>mainframe at the same time, simultaneously.

Yup.  We're using both and they work just fine.  Hopefully, someday WFWG
will communicate over LWP TCPIP.  Right now we have to load NetBeui.

I use ODI with ODINSUP and all works well.





DOS 등의 도구로 출력한 자료로 보인다.

1. '>' 같은 기호들 (단, 부등호는 조심할 것)

2. 연락처나 이름 같은 것들로 카테고리를 외운다면, 정답은 잘 맞추겠지만 과적합일 가능성이 높아질 것 같다. 또한 메일을 쓴 사람이 부서 이동을 할 수도 있으니. 이메일도 지우자.




DOS 등의 도구로 출력한 자료로 보인다.

1. 학습에 도움이 되지 않는 것들

    - \n, \t, |, > 등

    - 이메일

    - 누구누구 writes: / wrote:

    - Lines:




### 답장 구조

답장 구조가 존재한다.

답장 내용이 카테고리와 관련 없는 경우와 관련 있는 경우 모두 존재한다.

어떤 방향이든, 처음 쓰였던 내용이 계속 인용되어 여러 차례 학습에 노출된다는 점을 해결해야 한다.

In [21]:
## TODO: 답장 관련 내용 어떻게 처리할지 고민하기.

1. writes: wrote:로 1차로 답장 거름(quote_head)

2. 거기서 인용 부호 찾아내기

    - quote_head 다음 문장부터 인용 부호가 담긴 문장 찾기 -> quote_mark_sentence

        - 인용 부호 있으면 그대로 빼서 origin 찾기

        - 인용 부호 없으면 write: wrote: 없는 게 origin

In [22]:
import re

quote_head = r'.+?\s+(writes|wrote):\s*'

quote_mark_set_dict = dict()

for text_path in TRAIN_DICT.keys():

    text = get_text(text_path)
    quote_count = len([match.start() for match in re.finditer(quote_head, text)])

    if quote_count > 1:
        count = 0
        quote_mark_set_dict[text_path] = set()

        text_by_sentence = text.split("\n")
        for sentence in text_by_sentence:
            if re.search(quote_head, sentence):
                count += 1
                quote_mark = re.search(r"^\W*", sentence).group().strip()
                quote_mark_set_dict[text_path].add(quote_mark)

quote_mark_set_dict

{'/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.politics.mideast/76277': {'',
  '>('},
 '/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.politics.mideast/76045': {'',
  '|>'},
 '/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.politics.mideast/75406': {'',
  '|>'},
 '/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.politics.mideast/75401': {'',
  '|> ('},
 '/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.politics.mideast/76042': {''},
 '/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.politics.mideast/76270': {'',
  '>'},
 '/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.politics.mideast/76428': {'',
  '>',
  '>|>'},
 '/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.politics.mideast/76240': {'',
  '>'},
 '/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-byda

In [ ]:
# import re

# quote_head = r'.+?\s+(writes|wrote):\s*'
# ## TODO: In-Reply-To:
# ## TODO: As quoted from + 이메일
# ## TODO: Subject: Re:
# ## TODO: 이중 답장: quote_mark 반복 횟수로 판단. relation_dict를 0: 1:로 계층화 할 수 있을 것 같다.


# reply_dict = dict()

# for text_path in TRAIN_DICT.keys():
#     text = get_text(text_path)

#     quote_index_list = [match.start() for match in re.finditer(quote_head, text)] # writes: 등의 인용 표현 탐지

#     e_idx = 0

#     if len(quote_index_list) > 0:
#         enter_index_list = [match.start() for match in re.finditer("\n", text)] # 줄바꿈별 분리
        
#         reply_dict[text_path] = list()

#         for quote_index in quote_index_list:
#             # e_dix < q_idx < e_dix + 1 설정하기
#             while enter_index_list[e_idx + 1] <= quote_index:
#                 e_idx += 1

#             # quote_sentence 찾기
#             q_idx = e_idx + 1
#             quote_sentence = text[enter_index_list[q_idx] : enter_index_list[q_idx + 1]].strip("\n")

#             if re.search(r"\w", quote_sentence):
#                 quote_quote_mark = re.search(r"^\W*", quote_sentence).group().strip("\n").strip()

#             else:
#                 q_idx += 1
#                 quote_sentence = text[enter_index_list[q_idx] : enter_index_list[q_idx + 1]].strip("\n")
#                 quote_quote_mark = re.search(r"^\W*", quote_sentence).group().strip("\n").strip()


#             reply_dict[text_path].append({"quote_mark": quote_quote_mark,
#                                           "quote_sentence": quote_sentence})

In [25]:
# import json

# reply_dict_path = os.path.join(DATA_DIR, "reply_dict.json")

# with open(reply_dict_path, "w", encoding="utf-8") as f:
#     json.dump(reply_dict, f, indent=4)

In [36]:
import os
import json

def open_reply_dict():

    reply_dict_path = os.path.join(DATA_DIR, "reply_dict.json")

    with open(reply_dict_path, "r", encoding="utf-8") as f:
        reply_dict = json.load(f)
    
    return reply_dict

reply_dict = open_reply_dict()

In [37]:
yoju_set = {"/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/comp.sys.mac.hardware/50457",
             "/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/alt.atheism/54180",
             "/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.politics.misc/176895",
             "/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.politics.misc/178311",
             "/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.religion.misc/83689"}


for text_path, quote_dict_list in reply_dict.items():
    if text_path not in yoju_set:
        quote_mark_list = list({quote_dict["quote_mark"] for quote_dict in quote_dict_list})
        quote_mark_list = sorted(quote_mark_list, key=len)
        
        if len(quote_mark_list) > 1:
            for i in range(len(quote_mark_list)-1):
                if len(quote_mark_list[i]) == len(quote_mark_list[i+1]):
                    print(text_path)
                    print(quote_mark_list)

/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/rec.autos/101674
['>', '> <', '> >']
/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/comp.sys.mac.hardware/51880
['>', '}', '> |>', '> |> >']
/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/alt.atheism/51219
['>', ']']
/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.politics.misc/177001
['>', '>#', '>>']
/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/comp.sys.ibm.pc.hardware/60564
['>', ':']
/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/soc.religion.christian/21374
['>', '[']
/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.religion.misc/83602
['>', '[']


In [38]:
def list_quote_endswith(text: str):
    for text_path, quote_dict_list in reply_dict.items():
        if text_path not in yoju_set:
            quote_mark_list = list([quote_dict["quote_mark"], quote_dict["quote_sentence"]] 
                                   for quote_dict in quote_dict_list 
                                   if quote_dict["quote_mark"].endswith(text))

            if quote_mark_list != []:
                print(text_path)
                print(quote_mark_list)

In [39]:
yoju_set.update(["/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/alt.atheism/51318",
                 "/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.religion.misc/82795",
                 "/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.politics.mideast/75406",
                 "/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.politics.misc/176993",
                 "/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/soc.religion.christian/20709"
                 ])

list_quote_endswith("[")

/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/comp.windows.x/67253
[['[', '       [Hint for Sun OS users:  use /usr/5bin/echo instead of']]
/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/soc.religion.christian/21374
[['[', '[why are atheists atheists/ believes it could be the result of']]
/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.religion.misc/83602
[['[', '[ a lot of religious opinions and quotations from the Bible and from ']]


In [40]:
yoju_set.update(["/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/alt.atheism/51318",
                 "/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.religion.misc/82795",
                 ])

list_quote_endswith("(")

In [41]:
yoju_set.update(["/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/rec.autos/101572",
                 ])

list_quote_endswith("<")

/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.politics.mideast/75394
[['><', '><disclaimer: If there is anybody on USENET dumb enough to interpret']]
/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/rec.autos/101674
[['> <', "> <apparently you're not a woman - my husband hates the auto door locks"]]
/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/sci.crypt/15589
[['<', '<If Clipper comes to cellular phones along with legal proscriptions against']]
/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/sci.crypt/15667
[['<', '<the case that all the Republicans, etc. in the NSA and FBI and CIA']]
/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/sci.crypt/15555
[['<', '<As usually, you are not reading. The proposal -does- say that it is a']]
/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/sci.med/59153
[['<', '<The original poster mentioned fatigue.  I ha

In [42]:
list_quote_endswith("{")

In [ ]:
# yoju_list.extend["/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/alt.atheism/51318",
#                  "/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.religion.misc/82795",
#                  ]

# for text_path, quote_dict_list in reply_dict.items():
#     if text_path not in yoju_list:

#         for quote_dict in quote_dict_list:
#             if quote_dict["quote_mark"].endswith("("):
#                 quote_dict["quote_mark"] = quote_dict["quote_mark"].replace("(", "").strip()
#                 print(quote_dict)

In [144]:
related_path_dict = dict()

for path, quote_info in reply_dict.items():

    category = path.split("/")[-2]
    candidate_path_list = glob(os.path.join(TRAIN_DIR, category, "**"))
    
    for depth, quote_dict in quote_info.items():
        
        depth = depth
        quote_mark = quote_dict["quote_mark"]
        raw_quote = quote_dict["quote_sentence"]


        if quote_mark:
            processed_quote = raw_quote.split(quote_mark)[-1]
        else:
            processed_quote = raw_quote.strip()


        if processed_quote in related_path_dict.keys():
            related_path_dict[processed_quote]["reply"] = {depth: path}
        else:
            related_path_dict[processed_quote] = {"origin": set(), 
                                                "reply": {depth: path}}
            


        for candidate_path in candidate_path_list:
            
            if os.path.isfile(candidate_path):
                candidate_text = get_text(candidate_path)
                
                if processed_quote in candidate_text:
                    if raw_quote not in candidate_text:
                        related_path_dict[processed_quote]["origin"].add(candidate_path)
                    else:
                        related_path_dict[processed_quote]["reply"].add(candidate_path)

AttributeError: 'list' object has no attribute 'items'

In [ ]:
print("[related_path_dict 샘플 출력]")

i = 0
for key, value in related_path_dict.items():
    print(f"· key: {key}")
    print(f"· value: {value}")
    if value["origin"] != set():
        print(True)

[related_path_dict 샘플 출력]
· key: Lebanese resistance forces detonated a bomb under an Israeli occupation
· value: {'origin': {'/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.politics.mideast/75889', '/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.politics.mideast/75995'}, 'reply': {'/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.politics.mideast/75895'}}
True
· key: First of all I never said the Holocaust. I said before the
· value: {'origin': {'/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.politics.mideast/76223'}, 'reply': {'/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.politics.mideast/76253', '/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.politics.mideast/76296', '/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.politics.mideast/76248', '/Users/won/dev/00_codeit/0_mission/10_NLP_RN

In [ ]:
len(related_path_dict)

0

In [ ]:
len([raw_quote for raw_quote, relation_dict in related_path_dict.items() if len(relation_dict["origin"]) == 0])
len([raw_quote for raw_quote, relation_dict in related_path_dict.items() if len(relation_dict["origin"]) > 1])

0

In [ ]:
len([key for key, value in related_path_dict.items() if len(value["origin"]) > 1])
[key for key, value in related_path_dict.items() if len(value["origin"]) > 1]

[]

In [ ]:
def preprocess_text(text):
    """뉴스그룹 텍스트 정제 함수"""
    
    # # 1. 헤더 제거 (빈 줄 2개 전까지)
    # text = re.sub(r'^.*?\n\n', '', text, flags=re.DOTALL)
    
    # # 2. Article ID 제거 (<...@...> 형식)
    # text = re.sub(r'<[^>]+@[^>]+>', '', text)
    
    # # 3. 인용 표시 제거 (>, | 로 시작하는 줄)
    # quote_symbols = r'[>|]'
    # text = re.sub(rf'^\s*{quote_symbols}+.*$', '', text, flags=re.MULTILINE)
    
    # # 4. "writes:", "wrote:" 패턴 제거
    # text = re.sub(r'.+?\s+(writes|wrote):\s*', '', text, flags=re.IGNORECASE)
    
    ### [...]

    # 5. 메타데이터 라인 제거
    text = re.sub(r'^.*Lines:.*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'^.*From:.*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'^.*Organization:.*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'^.*Article-I.D.:.*$', '', text, flags=re.MULTILINE)
    # text = re.sub(r'^.*Distribution:.*$', '', text, flags=re.MULTILINE)
    # text = re.sub(r'^.*NNTP-Posting-Host:.*$', '', text, flags=re.MULTILINE)
    
    # # 6. 이메일 주소 제거
    # text = re.sub(r'\S+@\S+', '', text)
    
    # # 7. 서명 제거 (-- 이후)
    # text = re.sub(r'\n--+\s*\n.*$', '', text, flags=re.DOTALL)
    
    # # 8. URL 제거
    # text = re.sub(r'http[s]?://\S+|www\.\S+|ftp://\S+', '', text)
    
    # # 9. 연속 특수문자 제거
    # text = re.sub(r'[-=*]{3,}', '', text)
    
    # # 10. 빈 줄 정리 (테스트 단계에서는 \n 유지)
    # text = re.sub(r'\n{3,}', '\n\n', text)
    
    # # 11. 같은 줄 내 다중 공백만 제거 (\n은 유지)
    # text = re.sub(r'[ \t]+', ' ', text)
    
    # # 12. 앞뒤 공백 제거
    # text = text.strip()
    
    return text

print(preprocess_text(sample_text))


Subject: Re: Can sin "block" our prayers?



In article <Apr.7.23.20.24.1993.14263@athos.rutgers.edu>, 
	3225200@qucdn.queensu.ca writes:
> I have heard an interesting notion that sin can "block" our prayers to God,
> i.e. God will not hear our prayers if we have not confessed our sins. Now I am
> totally supportive of confessing our sins before God, but I simply do not
> believe God will "shut us out" just because we did not confess. This is kind of
> like the idea that suffering is caused by sin, which, as any Job reader will
> realize, is too simpilistic.
{rest deleted}

Can the Father possibly not hear the words of His children.
Of course He hears all your prayers.
Whether you are a sinner or a saint, no questions.
The real question you should be asking is: "Does sin block OUR hearing His
answer?" And the answer to that question is a resounding YES.
To paraphrase the gospel "Many are called but few choose to listen"
and so it is with prayer. 


In Christ,

James
-- 
______________

## 데이터셋 생성

In [ ]:
from torch.utils.data import Dataset

class TextDataset(Dataset):
    def __init__(self, dictionary):
        self.dictionary = dictionary
        self.text_path_list = list(self.dictionary.keys())

    def __len__(self):
        return len(self.text_path_list)
    
    def __getitem__(self, index):
        X_path = self.text_path_list[index]
        X = get_text(X_path)
        X = preprocess_text(X)
        y = self.dictionary[X_path]

        return X, y

train_dataset = TextDataset(dictionary=TRAIN_DICT)
test_dataset = TextDataset(dictionary=TEST_DICT)

train_dataset[0]

('\nSubject: Heil Hernlem \nIn-Reply-To: hernlem@chess.ncsu.edu\'s message of Wed, 14 Apr 1993 12:58:13 GMT\n\n\n\nIn article <1993Apr14.125813.21737@ncsu.edu> hernlem@chess.ncsu.edu (Brad Hernlem) writes:\n\n   Lebanese resistance forces detonated a bomb under an Israeli occupation\n   patrol in Lebanese territory two days ago. Three soldiers were killed and\n   two wounded. In "retaliation", Israeli and Israeli-backed forces wounded\n   8 civilians by bombarding several Lebanese villages. Ironically, the Israeli\n   government justifies its occupation in Lebanon by claiming that it is \n   necessary to prevent such bombardments of Israeli villages!!\n\n   Congratulations to the brave men of the Lebanese resistance! With every\n   Israeli son that you place in the grave you are underlining the moral\n   bankruptcy of Israel\'s occupation and drawing attention to the Israeli\n   government\'s policy of reckless disregard for civilian life.\n\n   Brad Hernlem (hernlem@chess.ncsu.EDU)\n\

In [ ]:
from torch.utils.data import WeightedRandomSampler
from torch.utils.data import DataLoader